# Diffusion Model: scoreでノイズから戻る

Diffusion Modelは、データへノイズを足す過程を用意し、その逆向きにデータらしい方向へ戻すモデルである。


## このノートの読み方

想定読者: 確率分布、正規分布、MSE、MLPの回帰を理解した学生。

MLPの次に読む教材として、直感、数式、shape、コード、HTMLアニメーション、`Trainer`学習例を往復しながら読む。


## MLPからの橋渡し

MLPは入力から正解へ直接写像した。Diffusionでは、壊れたデータから取り除くべきノイズ、または密度が高くなる方向scoreを学ぶ。


## 到達目標

- scoreが分類スコアではないことを説明できる
- forward noisingと逆向き学習を区別できる
- ノイズ予測Trainer例を読める


## 重要語句

- `score`: log densityの入力微分
- `denoising`: ノイズ成分を推定して戻すこと
- `Langevin`: score方向とランダム揺らぎで移動するサンプリング


## 準備

すべてのコードは小さなテンソルで概念を確認するためのものです。長い学習は行いません。


In [ ]:
from __future__ import annotations

import math

import numpy as np
import torch
from jaxtyping import Float
from torch import nn
from torch.utils.data import Dataset
import transformers
from transformers import Trainer, TrainingArguments

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)


## shape表

数式を読む前に、どのテンソルがどのshapeを持つかを固定する。

| 記号 | shape | 意味 |
|---|---|---|
| x0 | (B, dim) | きれいなデータ |
| sigma | (B, 1) | ノイズ量 |
| epsilon | (B, dim) | 混ぜたノイズ |


## レビュー指摘を踏まえた補強

| 観点 | 補足 |
|---|---|
| scoreとnoise | VE型の簡略式`x_t=x_0+sigma epsilon`では、最適なdenoising scoreは概ね`-(epsilon/sigma)`方向になる。ノイズ予測はscoreを扱いやすくした実装形である。 |
| DDPMとの差分 | この章はscoreとdenoisingの直感を扱う。DDPM章では`beta_t`と`alpha_bar_t`を使う離散時刻の実装へ落とす。 |
| 逆向き1ステップ | 予測したノイズを少し引き、必要ならランダム揺らぎを足して次の時刻へ進む。生成はこの小さな修正の反復である。 |
| 実データ例 | 蛍光顕微鏡画像の測定ノイズや分子構造生成では、ノイズを除きながらデータらしい構造へ戻す見方が役立つ。 |


## Forward noising

壊す過程は固定であり、学習しない。

$$
x_t=x_0+\sigma_t\epsilon,\quad \epsilon\sim\mathcal{N}(0,I)
$$


## Score

scoreは確率密度が高くなる方向を示すベクトルである。

$$
s_t(x)=\nabla_x\log p_t(x)
$$


## Denoising objective

実装ではscoreではなく混ぜたノイズを予測する形が扱いやすい。

$$
L=\mathbb{E}\| \epsilon-\epsilon_\theta(x_t,t)\|^2
$$


## 小さいテンソルで確認する

次のコードは、上の式がどのshapeを返すかを確認するための最小例である。


In [ ]:
x0 = torch.tensor([[1.0, -1.0], [0.8, -0.7]])
sigma = torch.tensor([[0.2], [0.8]])
epsilon = torch.randn_like(x0)
xt = x0 + sigma * epsilon
print("x0:", x0)
print("sigma:", sigma.squeeze())
print("xt:", xt.round(decimals=3))


## 難所HTMLスライド

数式だけでは混ざりやすい箇所を、スライド形式で確認する。各スライドでは入力shape、計算、lossまたは生成手順への接続を1つずつ見る。


<p><a href="../demos/diffusion-model_difficulty_slides.html?v=20260522" target="_blank" rel="noopener">別タブで難所スライドを開く</a>（リポジトリ内: <code>demos/diffusion-model_difficulty_slides.html</code>）</p>
<iframe
  src="../demos/diffusion-model_difficulty_slides.html?v=20260522"
  width="100%"
  height="720"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="Diffusion Model: scoreでノイズから戻る difficulty slides"
></iframe>


## HTMLアニメーションで確認する

以下のHTMLは`teaching-html-animation` skillの方針に合わせ、各状態を式・shape・コード上の概念に結びつけている。


### noising point cloud animation

- 学習目標: 点群がノイズへ崩れる
- 誤解の防止: forwardも学習すると思う

対応する式:

$$
x_t=x_0+\sigma_t\epsilon
$$


<p><a href="../demos/diffusion-model_noising_cloud.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/diffusion-model_noising_cloud.html</code>）</p>
<iframe
  src="../demos/diffusion-model_noising_cloud.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="noising point cloud animation"
></iframe>


### score field animation

- 学習目標: 密度が高い方向を矢印で示す
- 誤解の防止: scoreを分類スコアと思う

対応する式:

$$
s_t(x)=\nabla_x\log p_t(x)
$$


<p><a href="../demos/diffusion-model_score_field.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/diffusion-model_score_field.html</code>）</p>
<iframe
  src="../demos/diffusion-model_score_field.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="score field animation"
></iframe>


### Langevin decomposition animation

- 学習目標: score方向とランダム揺らぎを分ける
- 誤解の防止: 復元が決定的だと思う

対応する式:

$$
x_{k+1}=x_k+\eta s_\theta(x_k)+\sqrt{2\eta}z
$$


<p><a href="../demos/diffusion-model_langevin.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/diffusion-model_langevin.html</code>）</p>
<iframe
  src="../demos/diffusion-model_langevin.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="Langevin decomposition animation"
></iframe>


### denoising target animation

- 学習目標: モデルが予測するnoiseを見せる
- 誤解の防止: 画像そのものを暗記すると思う

対応する式:

$$
\epsilon_\theta(x_t,t)\approx\epsilon
$$


<p><a href="../demos/diffusion-model_denoise_target.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/diffusion-model_denoise_target.html</code>）</p>
<iframe
  src="../demos/diffusion-model_denoise_target.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="denoising target animation"
></iframe>


## `Trainer`で学習する

この章の`Trainer`例は、汎用MSE回帰ではなく、`Diffusion Model: scoreでノイズから戻る`固有のデータ形式とlossを返す。基本は標準の`Trainer(model, args, train_dataset)`を使い、`forward`が`loss`と`logits`を返す形にそろえる。GANはD/Gでoptimizerを分ける必要があるため`Trainer`を継承した交互更新デモ、DBMは平均場CDサロゲートとして扱う。


In [ ]:
class TinyDenoisingDataset(Dataset):
    def __init__(self, n_samples: int = 40, dim: int = 2) -> None:
        self.clean = torch.randn(n_samples, dim) * 0.4 + torch.tensor([1.0, -1.0])

    def __len__(self) -> int:
        return len(self.clean)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        clean = self.clean[index]
        sigma = torch.rand(1) * 0.8 + 0.1
        noise = torch.randn_like(clean)
        noisy = clean + sigma * noise
        return {"noisy": noisy, "sigma": sigma, "noise": noise}


class TinyDenoisingModel(nn.Module):
    def __init__(self, dim: int = 2) -> None:
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim + 1, 32), nn.SiLU(), nn.Linear(32, dim))

    def forward(self, noisy: torch.Tensor, sigma: torch.Tensor, noise: torch.Tensor | None = None) -> dict[str, torch.Tensor]:
        pred_noise = self.net(torch.cat([noisy, sigma], dim=-1))
        pred_score = -pred_noise / sigma.clamp_min(1e-4)
        loss = torch.mean((pred_noise - noise) ** 2) if noise is not None else None
        return {"loss": loss, "logits": pred_noise, "score": pred_score}


training_args = TrainingArguments(
    output_dir="./results/diffusion-model_trainer_demo",
    max_steps=3,
    per_device_train_batch_size=8,
    learning_rate=1e-3,
    logging_strategy="no",
    save_strategy="no",
    report_to="none",
    disable_tqdm=True,
    seed=SEED,
    use_cpu=not torch.cuda.is_available(),
)

trainer = Trainer(model=TinyDenoisingModel(), args=training_args, train_dataset=TinyDenoisingDataset())
train_output = trainer.train()
sample = TinyDenoisingDataset(n_samples=1)[0]
with torch.no_grad():
    out = trainer.model(sample["noisy"].unsqueeze(0), sample["sigma"].unsqueeze(0))
    one_step = sample["noisy"].unsqueeze(0) - sample["sigma"].unsqueeze(0) * out["logits"]
print("Denoising Trainer loss:", train_output.training_loss)
print("one denoise step:", one_step.round(decimals=3))


## 生成モデル間の比較

| モデル | 学習目的 | 尤度 | 生成手順 | 代表的な弱点 |
|---|---|---|---|---|
| VAE/IWAE | ELBO / IWAE bound | 下界 | Decoderにzを入れる | ぼやけ、推論分布の設計 |
| GAN | Dをだます | 通常は不可 | G(z)を一発生成 | mode collapse、不安定 |
| Flow | NLL | 厳密 | 可逆変換の順方向 | 可逆層の制約 |
| RBM/DBM | エネルギー差 | 分配関数が困難 | Gibbs sampling | 近似推論が重い |
| Diffusion/DDPM | score/noise予測 | 目的により異なる | 多段denoising | samplingが遅い |


## 発展課題

- 2D点群でscore場を描く
- Langevin更新を数ステップ実装する
- DDPM章で離散時刻へ進む


## 確認問題

- scoreは分類スコアと何が違うか。
- forward noisingは学習対象か。


## まとめ

- MLPから何が変わったのかを、shapeとlossで確認する。
- HTMLアニメーションは式の代わりではなく、式とコードを読むための補助である。
- `Trainer`は学習ループを隠すが、`Dataset`のキー、`forward`の引数、`loss`の意味は必ず確認する。
